# 04 — Pull DEMs from Earth Engine

**Purpose.** Downloads one elevation model per glacier: NASADEM (`NASA/NASADEM_HGT/001`) clipped to
the glacier's bounding box at 30 m. Elevation is the model's 9th input channel. It doesn't change over
time, so one DEM serves every date of a glacier's series (`config.dem_path`).

Set `LABEL` to choose the dataset:
- `config.INFERENCE_DATA_LABEL` (default): the 1,024 inference glaciers (250 per region), as in
  notebook 05.
- `config.TRAINING_DATA_LABEL`: every selected glacier. The training DEMs on disk came from an older
  notebook (`ee_pull_dems.ipynb`, commit `b801e42`) and have no S3 copy; this rebuilds them.

Glaciers that already have a DEM are skipped.

## Inputs
- **`config.inference_sample_path(250)`** (inference only): the glacier list, the saved Athena output
  of `sql/identify_inference_glims_ids_geog_area_rollup_250.sql`. Gitignored.
- **`config.glims_selection_path()`**: bounding boxes from notebook 01.
- **Earth Engine account**, as in notebook 03.

## Outputs
- `config.dem_dir(LABEL)/<glims_id>_NASADEM.tif`, one per glacier, flat. Gitignored.

## Manual step afterwards (not in code)
Inference DEMs belong in `s3://full-time-series-dems-t1-l2-sr` (**us-east-1**), flat. There is no
training DEM bucket. See `.agents/references/aws.md`.

In [ ]:
import ee
import pandas as pd
from tqdm import tqdm

from glacierview import config, glims, log
from glacierview.earthengine import EePull

log.configure()
ee.Authenticate()
ee.Initialize(project="psyched-upgrade-377000")

In [ ]:
LABEL = config.INFERENCE_DATA_LABEL

glaciers = glims.read_selection()
if LABEL == config.INFERENCE_DATA_LABEL:
    chosen = pd.read_csv(config.inference_sample_path(250))["glims_id"]
    glaciers = glaciers[glaciers["glac_id"].isin(chosen)]
glaciers = glaciers[[not config.dem_path(g, LABEL).exists() for g in glaciers["glac_id"]]]
pull = EePull(config.log_dir(LABEL), "dems")
print(f"{len(glaciers)} DEMs to pull")

In [ ]:
for glacier in tqdm(glaciers.itertuples(), total=len(glaciers)):
    bounding_box = glims.parse_bounding_box(glacier.bboxes)
    pull.export_dem(glacier.glac_id, bounding_box, config.dem_dir(LABEL))